# Notebook 3 — One training step

After [Notebook 2 — Neural networks](02_neural_network.ipynb), you can define a net and run `forward`. This notebook adds the missing link: sample → score → loss → update weights.

We stay on tiny fake tensors. No game environment, no learning from real episodes yet.

Use the `pong-pytorch` kernel.

## Running this notebook

Locally, the environment comes from the repo — `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

On Colab, the kernel exists before the project does, so the cell below
clones the repo and installs what Colab lacks. Run it first.


In [ ]:
# Colab bootstrap. No-op when running locally.
import pathlib
import sys

IN_COLAB = "google.colab" in sys.modules

# False: install only what Colab lacks (seconds).
# True:  install the locked environment exactly, which is reproducible but
#        downloads ~1GB and replaces the torch build Colab ships with.
EXACT_ENV = False

if IN_COLAB:
    if not pathlib.Path("/content/pong-pytorch").exists():
        !git clone -q https://github.com/piyushahuja/pong-pytorch.git /content/pong-pytorch
    %cd /content/pong-pytorch

    if EXACT_ENV:
        !pip install -q uv
        !uv export --locked --no-dev --format requirements.txt -o /tmp/pong-req.txt
        !pip install -q -r /tmp/pong-req.txt
    else:
        !pip install -q gymnasium ale-py  # torch and matplotlib ship with Colab

    # Puts the pong package on the path without touching its dependencies.
    !pip install -q -e . --no-deps

    print("bootstrapped in", pathlib.Path.cwd())
else:
    print("local run: using the uv environment from uv.lock")


In [ ]:
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli

print(torch.__version__)
import pong        # PROJECT_ROOT: paths that work from anywhere


## From a probability to a choice

Suppose the network outputs a single number $p \in (0,1)$ (Notebook 2’s sigmoid head).

A Bernoulli distribution treats $p$ as “chance of 1” (vs 0).

- `sample()` — draw 0 or 1
- `log_prob(action)` — how unsurprising that draw was under $p$ (natural log)

You will store `log_prob` for whatever you actually sampled.

In [ ]:
p = torch.tensor(0.7)                    # pretend this came from a network
dist = Bernoulli(probs=p)

torch.manual_seed(0)
action = dist.sample()
lp = dist.log_prob(action)

print("p:", p.item())
print("sampled action:", action.item(), "(1 means 'option A', 0 means 'option B')")
print("log_prob:", lp.item())
print("prob of that action:", lp.exp().item())

Several steps → several log-probs. Stack them into one tensor.

In [ ]:
torch.manual_seed(1)
ps = torch.tensor([0.2, 0.5, 0.9])
dist = Bernoulli(probs=ps)
actions = dist.sample()
log_probs = dist.log_prob(actions)

print("p:        ", ps)
print("actions:  ", actions)
print("log_probs:", log_probs)
print("stacked shape:", log_probs.shape)

## Weight those log-probs

Idea: if a sequence of choices looked good, make those choices a bit more probable; if bad, a bit less.

A simple scalar loss that does that (gradient descent *minimizes* loss):

$$
L = -\sum_t w_t \, \log \pi(a_t)
$$

Here $w_t$ are made-up weights (later: discounted returns). Positive $w$ pushes up $\log\pi$; negative $w$ pushes it down.

In [ ]:
weights = torch.tensor([1.0, -0.5, 2.0])   # fake "how good was this step?"
loss = -(log_probs * weights).sum()
print("loss:", loss.item())

## One real update: forward → loss → backward → step

Wire a tiny net into the same pattern.

1. `y = model(x)` → sigmoid → $p$
2. sample + `log_prob`
3. build `loss`
4. `loss.backward()` writes gradients into `.grad`
5. `optimizer.step()` changes weights
6. `optimizer.zero_grad()` clears grads for next time

In [ ]:
class TinyBinaryNet(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=16):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        return torch.sigmoid(self.layer2(x)).squeeze(-1)


torch.manual_seed(0)
model = TinyBinaryNet()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)

x = torch.randn(8)
w_before = model.layer2.weight.detach().clone()

p = model(x)
dist = Bernoulli(probs=p)
action = dist.sample()
log_prob = dist.log_prob(action)

# pretend this choice was "good"
weight = torch.tensor(1.0)
loss = -(log_prob * weight)

optimizer.zero_grad()
loss.backward()
optimizer.step()

w_after = model.layer2.weight.detach()
print("p:", p.item(), "action:", action.item(), "loss:", loss.item())
print("layer2 weight changed:", not torch.allclose(w_before, w_after))
print("max |ΔW|:", (w_after - w_before).abs().max().item())

## Discounted returns from a reward list

Weights $w_t$ often come from future rewards. Walking backward:

$$
R_t = r_t + \gamma R_{t+1}
$$

with $R_{T+1} = 0$. So early steps feel the later $+1$ / $-1$, faded by $\gamma$.

In [ ]:
def discount_rewards(rewards, gamma=0.99):
    """rewards: list of floats, e.g. [0, 0, 0, 1]."""
    discounted = [0.0] * len(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = running * gamma + rewards[t]
        discounted[t] = running
    return torch.tensor(discounted, dtype=torch.float32)


rewards = [0.0, 0.0, 0.0, 1.0]
returns = discount_rewards(rewards, gamma=0.99)
print("rewards:", rewards)
print("returns:", returns.tolist())

Often we normalize returns (zero mean, unit variance) so updates are not dominated by raw scale.

In [ ]:
def normalize(returns):
    return (returns - returns.mean()) / (returns.std(unbiased=False) + 1e-8)


normed = normalize(returns)
print("normalized:", normed.tolist())
print("mean~", normed.mean().item(), "std~", normed.std(unbiased=False).item())

### Optional: reset the running return on nonzero rewards

In some games a nonzero reward ends a *sub*-episode (a point). You can zero the running return there so credit does not leak across points.

In [ ]:
def discount_rewards_reset_on_score(rewards, gamma=0.99):
    discounted = [0.0] * len(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        if rewards[t] != 0:
            running = 0.0
        running = running * gamma + rewards[t]
        discounted[t] = running
    return torch.tensor(discounted, dtype=torch.float32)


rewards2 = [0.0, 0.0, 1.0, 0.0, 0.0, -1.0]
print("plain :", discount_rewards(rewards2).tolist())
print("reset :", discount_rewards_reset_on_score(rewards2).tolist())

## Put it together (still fake data)

One “episode”: several forward passes, collect log-probs and rewards, discount, one backward.

In [ ]:
torch.manual_seed(2)
model = TinyBinaryNet()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)

log_probs = []
rewards = []

for t in range(5):
    x = torch.randn(8)
    p = model(x)
    dist = Bernoulli(probs=p)
    action = dist.sample()
    log_probs.append(dist.log_prob(action))
    # fake rewards: last step gets +1
    rewards.append(1.0 if t == 4 else 0.0)

returns = normalize(discount_rewards(rewards))
episode_log_probs = torch.stack(log_probs)

loss = -(episode_log_probs * returns).sum()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("rewards:", rewards)
print("returns (norm):", returns.tolist())
print("loss:", loss.item())

## Optional reading: same shapes as Notebook 2’s game examples

Still no environment — only “if $p$ came from that kind of head…”.

| Shape idea | $p$ / logits | Sampling |
|---|---|---|
| Pong-like | one sigmoid $p$ | `Bernoulli(p)` → 0/1 |
| Chess / Go-like | many logits → softmax | sample a category (not shown here) |

This notebook practiced the Bernoulli path because it matches a single probability output.

In [ ]:
# Pong-shaped head: 6400 -> 1 probability (fake frame)
pong_head = nn.Sequential(
    nn.Linear(6400, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid(),
)
fake_frame = torch.randn(6400)
p = pong_head(fake_frame).squeeze()
action = Bernoulli(probs=p).sample()
print("pong-like p", p.item(), "sample", action.item())

## Grad accumulation (concept)

If you call `backward()` several times without `zero_grad()`, gradients add up. That is useful for “update once every $N$ episodes.” Pattern:

```text
zero_grad
for episode in 1..N:
    ... compute loss ...
    loss.backward()      # accumulates
optimizer.step()
zero_grad
```

In [ ]:
torch.manual_seed(3)
model = TinyBinaryNet()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)

optimizer.zero_grad()
for _ in range(3):
    p = model(torch.randn(8))
    lp = Bernoulli(probs=p).log_prob(Bernoulli(probs=p).sample())
    loss = -lp * 1.0
    loss.backward()                      # accumulate

grad_norm = model.layer2.weight.grad.norm().item()
optimizer.step()
optimizer.zero_grad()
print("grad norm after 3 accumulated backwards:", grad_norm)

## Checkpoint after an update

In [ ]:
ckpt = pong.PROJECT_ROOT / "assets" / "_demo_train_step.pt"   # not Path("assets"): the notebook is not the root
torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    },
    ckpt,
)
print("saved", ckpt)

## What you can do now

| Skill | API |
|---|---|
| Sample a binary choice | `Bernoulli(probs=p).sample()` |
| Score it | `.log_prob(action)` |
| Scalar objective | `-(log_probs * weights).sum()` |
| Update weights | `backward` → `optimizer.step` → `zero_grad` |
| Turn rewards into weights | `discount_rewards` (+ normalize) |

Still ahead (later notebook / main project): turn pixels into the input vector, step a real environment, and run this loop for many episodes.

You now have the train-step core: probability → sample → log_prob → weighted loss → optimizer.